# 🚀 50M Bengali GPT — সম্পূর্ণ GGUF, INT4 এবং ONNX এক্সপোর্ট ও টেস্ট
### 🎯 যা যা তৈরি হবে:
1. **`bengali_gpt_50m_q4.gguf`** (INT4 Q4_K_M — মাত্র ~৩২ MB, `ffn_gate` ও `BPE merges` সহ ১০০% ফিক্সড)
2. **`bengali_gpt_50m_f16.gguf`** (FP16 Master GGUF — ~২০৮ MB)
3. **`bengali_gpt_50m.onnx`** (ONNX — itel A60 সহ যেকোনো 32-bit ও 64-bit অ্যান্ড্রয়েড ফোনে সরাসরি সাপোর্ট)
4. **`model.safetensors`** (Hugging Face ফরম্যাট — ~২০৭ MB)
5. **🧪 লাইভ টেস্ট:** এক্সপোর্টের পর Colab-এই `llama-cli` দিয়ে মডেল চালিয়ে আসল আউটপুট টেস্ট করা হবে!


In [ ]:
# Step 1: Google Drive মাউন্ট করুন
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BASE_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
EXPORT_DIR = os.path.join(DRIVE_BASE_DIR, 'exported_model')
os.makedirs(EXPORT_DIR, exist_ok=True)
print('✓ Google Drive মাউন্ট সফল!')

In [ ]:
# Step 2: রিপোজিটরি ক্লোন ও লাইব্রেরি ইনস্টল
%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git
%cd /content/ss_100m/ss_50million
!pip install -q safetensors gguf onnx onnxruntime onnxscript
print('✓ প্রয়োজনীয় লাইব্রেরি ইনস্টল সম্পন্ন!')

In [ ]:
# Step 3: 📦 সম্পূর্ণ এক্সপোর্ট (Safetensors + GGUF + INT4 + ONNX)
import os, sys, shutil, subprocess, json, torch
from scripts.export_model import export_to_safetensors, export_to_gguf, export_to_onnx

# চেকপয়েন্ট নির্বাচন
checkpoint_final = os.path.join(DRIVE_BASE_DIR, 'checkpoint_stage_2_final.pt')
if not os.path.exists(checkpoint_final):
    checkpoint_final = os.path.join(DRIVE_BASE_DIR, 'checkpoint_stage_1.pt')
print(f'📥 সোর্স চেকপয়েন্ট: {checkpoint_final}')

# ১. Safetensors এক্সপোর্ট
safetensors_path = export_to_safetensors(checkpoint_final, os.path.join(EXPORT_DIR, 'safetensors'))

# ২. GGUF FP16 এক্সপোর্ট (ffn_gate এবং BPE merges সহ ফিক্সড)
local_f16 = '/content/bengali_gpt_50m_f16.gguf'
if not os.path.exists(local_f16):
    export_to_gguf(checkpoint_final, local_f16, 'tokenizer.json')
shutil.copy(local_f16, os.path.join(EXPORT_DIR, 'bengali_gpt_50m_f16.gguf'))

# ৩. ONNX এক্সপোর্ট (itel A60 32-bit সাপোর্টের জন্য)
try:
    local_onnx = '/content/bengali_gpt_50m.onnx'
    export_to_onnx(checkpoint_final, local_onnx)
    if os.path.exists(local_onnx):
        shutil.copy(local_onnx, os.path.join(EXPORT_DIR, 'bengali_gpt_50m.onnx'))
except Exception as err:
    print(f'⚠️ ONNX এক্সপোর্ট স্কিপ করা হয়েছে: {err}')

# ৪. llama.cpp কম্পাইল করে INT4 (Q4_K_M) তৈরি করা
print('\n⚡ INT4 কোয়ান্টাইজেশন তৈরি হচ্ছে (llama.cpp)...')
if not os.path.exists('/content/llama_cpp_build'):
    os.system('git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama_cpp_build')
os.system('cd /content/llama_cpp_build && cmake -B build && cmake --build build --config Release -t llama-quantize llama-cli -j4')

quant_bin = '/content/llama_cpp_build/build/bin/llama-quantize'
if not os.path.exists(quant_bin):
    quant_bin = '/content/llama_cpp_build/llama-quantize'

local_q4 = '/content/bengali_gpt_50m_q4.gguf'
if os.path.exists(quant_bin):
    os.system(f'{quant_bin} {local_f16} {local_q4} q4_k_m')
    if os.path.exists(local_q4):
        shutil.copy(local_q4, os.path.join(EXPORT_DIR, 'bengali_gpt_50m_q4.gguf'))
        q4_mb = os.path.getsize(local_q4) / (1024 * 1024)
        print(f'🎉 INT4 GGUF প্রস্তুত! সাইজ: {q4_mb:.1f} MB')
        print(f'💾 ড্রাইভে সংরক্ষিত: {os.path.join(EXPORT_DIR, "bengali_gpt_50m_q4.gguf")}')
print('\n✅ সব ফরম্যাট এক্সপোর্ট সম্পন্ন!')

In [ ]:
# Step 4: 🧪 Colab-এ তৈরি হওয়া GGUF মডেল লাইভ টেস্ট করুন!
import os
cli_bin = '/content/llama_cpp_build/build/bin/llama-cli'
model_q4 = '/content/bengali_gpt_50m_q4.gguf'

if os.path.exists(cli_bin) and os.path.exists(model_q4):
    print('=' * 65)
    print('🤖 GGUF মডেল টেস্ট চলছে (llama-cli দিয়ে)...')
    print('=' * 65)
    # একটি টেস্ট প্রশ্ন রান করুন:
    !{cli_bin} -m {model_q4} -p "প্রশ্ন: ডিজিটাল মার্কেটিং কী? উত্তর:" -n 120 --temp 0.7
else:
    print('⚠️ ফাইল প্রস্তুত হয়নি, দয়া করে আগে Step 3 সম্পন্ন করুন।')

In [ ]:
# Step 5: ⬇️ ব্রাউজারে সরাসরি ডাউনলোড করুন (Download to Computer)
from google.colab import files
import os

q4_path = '/content/bengali_gpt_50m_q4.gguf'
if os.path.exists(q4_path):
    print(f'⬇️ আপনার কম্পিউটারে GGUF INT4 ডাউনলোড হচ্ছে ({os.path.getsize(q4_path)/(1024*1024):.1f} MB)...')
    files.download(q4_path)
else:
    print('⚠️ ফাইল পাওয়া যায়নি!')